# 🇰🇷 국내 주식 멀티팩터 밸류에이션 스크리너
**Korea Multi-Factor Valuation Screener**

| 항목 | 내용 |
|------|------|
| 데이터 소스 | NAVER Finance (크롤링), DART Open API |
| 대상 시장 | KOSPI, KOSDAQ |
| 수집 지표 | PER, PBR, ROE, EV/EBITDA, 시가총액 |
| 스크리닝 방식 | Z-score 멀티팩터 종합점수 |

**분석 프로세스**
1. NAVER Finance → 전체 상장 종목 PER·ROE·시가총액 수집
2. 조건 기반 1차 스크리닝
3. NAVER 개별 종목 페이지 → PBR 수집
4. DART Open API → EV/EBITDA 산출
5. Z-score 종합점수 산출 및 시각화

> **주의**: DART API 키가 필요합니다. [dart.fss.or.kr](https://dart.fss.or.kr) 에서 무료 발급 가능합니다.

---

## 0. 환경 설정

In [1]:
# 필요 라이브러리 설치 (최초 1회, 주석 해제 후 실행)
!apt-get install -y fonts-nanum
!pip install finance-datareader plotly beautifulsoup4

In [3]:
import requests
import pandas as pd
import numpy as np
import time
import os
import zipfile
import io
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.font_manager as fm
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from bs4 import BeautifulSoup
import FinanceDataReader as fdr
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
font_prop = fm.FontProperties(fname=font_path)
plt.rcParams['font.family'] = font_prop.get_name()
plt.rcParams['axes.unicode_minus'] = False
print('라이브러리 로드 완료 ✓')

KeyboardInterrupt: 

In [ ]:
# DART API 키 설정 (보안)
# 방법 1: 환경변수 (권장)
DART_API_KEY = os.environ.get('DART_API_KEY', '')

# 방법 2: Colab Secrets
if not DART_API_KEY:
    try:
        from google.colab import userdata
        DART_API_KEY = userdata.get('DART_API_KEY')
    except:
        pass

# 방법 3: 직접 입력
if not DART_API_KEY:
    DART_API_KEY = input('DART API Key 입력: ')

print('API 키 설정 완료 ✓')

## 1. NAVER Finance 크롤링
KOSPI·KOSDAQ 전체 상장 종목의 PER, ROE, 시가총액을 수집합니다.

In [ ]:
def crawl_naver_market(sosok: int, market_name: str) -> pd.DataFrame:
    headers = {'User-Agent': 'Mozilla/5.0'}
    all_data = []
    page = 1
    while True:
        url = f'https://finance.naver.com/sise/sise_market_sum.naver?sosok={sosok}&page={page}'
        resp = requests.get(url, headers=headers)
        tables = pd.read_html(resp.text, encoding='euc-kr')
        df = tables[1].dropna(subset=['종목명'])
        if len(df) == 0:
            break
        all_data.append(df)
        print(f'  [{market_name}] {page}페이지 ({len(df)}개)', end='\r')
        page += 1
        time.sleep(0.3)
    result = pd.concat(all_data, ignore_index=True)
    result['시장'] = market_name
    print(f'\n  [{market_name}] 완료: {len(result)}개')
    return result

print('=== NAVER Finance 크롤링 시작 ===')
kospi_raw  = crawl_naver_market(0, 'KOSPI')
kosdaq_raw = crawl_naver_market(1, 'KOSDAQ')
raw_df = pd.concat([kospi_raw, kosdaq_raw], ignore_index=True)
print(f'전체 수집: {len(raw_df):,}개 종목')

## 2. 전처리

In [ ]:
df = raw_df[['종목명', '시장', '시가총액', 'PER', 'ROE']].copy()
for col in ['시가총액', 'PER', 'ROE']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# FDR에서 시가총액(원 단위) 보완
kospi_fdr  = fdr.StockListing('KOSPI')[['Name', 'Marcap']]
kosdaq_fdr = fdr.StockListing('KOSDAQ')[['Name', 'Marcap']]
fdr_df = pd.concat([kospi_fdr, kosdaq_fdr]).rename(columns={'Name': '종목명', 'Marcap': '시가총액_원'})
df = df.merge(fdr_df, on='종목명', how='left')
df['시가총액_억'] = df['시가총액_원'] / 1e8

# 이상치 제거
before = len(df)
df = df[(df['PER'] > 0) & (df['ROE'] > 0) & (df['시가총액_억'] > 0)].copy()
for col in ['PER', 'ROE']:
    lo, hi = df[col].quantile([0.01, 0.99])
    df = df[df[col].between(lo, hi)]

print(f'전처리: {before:,}개 → {len(df):,}개 종목')
df[['PER', 'ROE', '시가총액_억']].describe().round(2)

## 3. 1차 스크리닝
PER, ROE, 시가총액 기준으로 종목을 필터링합니다.

In [ ]:
def screen_stocks(df, per_max=15, roe_min=10, mktcap_min=500, market=None):
    r = df.copy()
    if market:
        r = r[r['시장'] == market]
    r = r[
        (r['PER'] <= per_max) &
        (r['ROE'] >= roe_min) &
        (r['시가총액_억'] >= mktcap_min)
    ].copy()
    return r[['종목명', '시장', 'PER', 'ROE', '시가총액_억']].reset_index(drop=True)

# 스크리닝 조건 (필요 시 조정)
result = screen_stocks(df, per_max=15, roe_min=10, mktcap_min=500)
print(f'1차 스크리닝 통과: {len(result)}개 종목')
result.head(5)

## 4. PBR 수집
NAVER Finance 개별 종목 페이지에서 PBR을 수집합니다.
PBR = 현재주가 / BPS (최근 분기 자본총계 기준)

In [ ]:
# 종목코드 매핑 테이블 생성
kospi_list  = fdr.StockListing('KOSPI')[['Code', 'Name']]
kosdaq_list = fdr.StockListing('KOSDAQ')[['Code', 'Name']]
code_map = pd.concat([kospi_list, kosdaq_list])
code_map.columns = ['종목코드', '종목명']

def get_naver_pbr(code: str) -> float:
    url = f'https://finance.naver.com/item/main.naver?code={code}'
    headers = {'User-Agent': 'Mozilla/5.0'}
    resp = requests.get(url, headers=headers)
    soup = BeautifulSoup(resp.text, 'html.parser')
    for th in soup.find_all('th'):
        if 'PBR' in th.get_text() and 'span' in str(th):
            td = th.find_next_sibling('td')
            if td:
                try:
                    return float(td.get_text().strip().replace(',', ''))
                except:
                    return None
    return None

result_pbr = result.merge(code_map, on='종목명', how='left')
print(f'PBR 수집 시작 (약 {len(result_pbr)//3}분 소요)...')
pbr_list = []
for i, row in result_pbr.iterrows():
    code = row['종목코드']
    if pd.isna(code):
        pbr_list.append(None)
        continue
    pbr_list.append(get_naver_pbr(code))
    if len(pbr_list) % 50 == 0:
        print(f'  {len(pbr_list)}/{len(result_pbr)} 완료...')
    time.sleep(0.4)

result_pbr['PBR'] = pbr_list
print(f'PBR 확보: {result_pbr["PBR"].notna().sum()}/{len(result_pbr)}개')

## 5. EV/EBITDA 수집 (DART Open API)

**EV/EBITDA 계산 방식**
- EV = 시가총액 + 부채총계 - 현금및현금성자산
- EBITDA ≈ 영업활동현금흐름 + 법인세납부(현금유출분) + 이자지급

**주의사항**
- DART 공시 계정과목명은 기업마다 비표준화되어 있어 복수의 후보명으로 탐색
- 법인세환급(납부) 항목은 부호 방향에 따라 처리 (음수=납부, 양수=환급)
- 영업활동현금흐름 음수 또는 데이터 누락 시 None 처리

In [ ]:
# DART 기업 고유코드 목록 다운로드
print('DART 기업코드 목록 다운로드 중...')
resp = requests.get(
    'https://opendart.fss.or.kr/api/corpCode.xml',
    params={'crtfc_key': DART_API_KEY}
)
z = zipfile.ZipFile(io.BytesIO(resp.content))
tree = ET.fromstring(z.read('CORPCODE.xml'))

corp_list = []
for item in tree.findall('list'):
    corp_list.append({
        'corp_code':  item.findtext('corp_code'),
        'corp_name':  item.findtext('corp_name'),
        'stock_code': item.findtext('stock_code'),
    })
corp_df = pd.DataFrame(corp_list)
corp_df = corp_df[corp_df['stock_code'].str.strip() != '']
print(f'상장 기업 수: {len(corp_df)}개 ✓')

In [ ]:
def get_dart_financials(corp_code: str, api_key: str) -> dict:
    """
    DART API에서 EV/EBITDA 계산용 재무데이터 수집
    연결재무제표(CFS) 우선, 없으면 별도재무제표(OFS) 사용

    EBITDA 근사 = 영업활동현금흐름 + 법인세납부(현금유출분) + 이자지급
    DART 계정과목명은 기업마다 비표준화 → 복수 후보명으로 탐색
    """
    url = 'https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json'
    params = {
        'crtfc_key': api_key,
        'corp_code': corp_code,
        'bsns_year': '2024',
        'reprt_code': '11011',
        'fs_div': 'CFS'
    }
    resp = requests.get(url, params=params)
    data = resp.json()
    if data['status'] != '000':
        params['fs_div'] = 'OFS'
        data = requests.get(url, params=params).json()
    if data['status'] != '000':
        return None

    df = pd.DataFrame(data['list'])

    def get_val(candidates, sj_nm=None):
        if isinstance(candidates, str):
            candidates = [candidates]
        for acc in candidates:
            mask = df['account_nm'] == acc
            if sj_nm:
                mask &= df['sj_nm'] == sj_nm
            row = df[mask]
            if len(row) > 0:
                try:
                    return float(str(row.iloc[0]['thstrm_amount']).replace(',', ''))
                except:
                    continue
        return None

    영업활동현금흐름 = get_val('영업활동현금흐름', '현금흐름표')
    법인세납부 = get_val([
        '법인세 납부액', '법인세의 납부', '법인세납부(환급)',
        '법인세납부액', '법인세 환급(납부)', '법인세의납부',
        '법인세납부', '법인세환급(납부)',
    ], '현금흐름표')
    이자지급 = get_val([
        '이자의 지급', '이자지급(영업)', '이자의지급',
        '이자 지급', '이자지급액', '이자지급',
    ], '현금흐름표')
    부채총계 = get_val('부채총계', '재무상태표')
    현금     = get_val('현금및현금성자산', '재무상태표')

    try:
        # 법인세: 음수(납부)일 때만 더함, 양수(환급)는 제외
        tax_add = abs(법인세납부) if (법인세납부 is not None and 법인세납부 < 0) else 0
        interest_add = abs(이자지급) if 이자지급 is not None else 0
        ebitda = 영업활동현금흐름 + tax_add + interest_add
    except:
        ebitda = None

    return {'부채총계': 부채총계, '현금': 현금, 'EBITDA': ebitda}


# corp_code 매핑 및 EV/EBITDA 수집
dart_map = corp_df[['corp_code', 'stock_code']].copy()
dart_map.columns = ['corp_code', '종목코드']
result_final = result_pbr.merge(dart_map, on='종목코드', how='left')

print(f'EV/EBITDA 수집 시작 (약 {len(result_final)//6}분 소요)...')
ev_ebitda_list = []
for i, row in result_final.iterrows():
    corp_code = row['corp_code']
    mktcap_원 = row['시가총액_억'] * 1e8
    if pd.isna(corp_code):
        ev_ebitda_list.append(None)
        continue
    fin = get_dart_financials(corp_code, DART_API_KEY)
    try:
        if fin is None or fin['EBITDA'] is None or fin['EBITDA'] <= 0:
            ev_ebitda_list.append(None)
        else:
            ev = mktcap_원 + fin['부채총계'] - fin['현금']
            ev_ebitda_list.append(round(ev / fin['EBITDA'], 2))
    except:
        ev_ebitda_list.append(None)
    if len(ev_ebitda_list) % 50 == 0:
        print(f'  {len(ev_ebitda_list)}/{len(result_final)} 완료...')
    time.sleep(0.5)

result_final['EV/EBITDA'] = ev_ebitda_list
success = result_final['EV/EBITDA'].notna().sum()
print(f'EV/EBITDA 확보: {success}/{len(result_final)}개')
print(f'나머지 {len(result_final)-success}개는 DART 데이터 부재 또는 EBITDA 음수로 None 처리')

## 6. 종합점수 산출
각 지표를 Z-score로 표준화 후 합산합니다.
- PER, PBR, EV/EBITDA: 낮을수록 좋음 → 음(-)의 Z-score
- ROE: 높을수록 좋음 → 양(+)의 Z-score
- EV/EBITDA 확보 종목: 4개 지표, 미확보 종목: 3개 지표

In [ ]:
def calc_final_score(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['z_per'] = -((df['PER'] - df['PER'].mean()) / df['PER'].std())
    df['z_pbr'] = -((df['PBR'] - df['PBR'].mean()) / df['PBR'].std())
    df['z_roe'] =  (df['ROE'] - df['ROE'].mean()) / df['ROE'].std()
    has_ev = df['EV/EBITDA'].notna()
    df.loc[has_ev, 'z_ev'] = -(
        (df.loc[has_ev, 'EV/EBITDA'] - df.loc[has_ev, 'EV/EBITDA'].mean()) /
        df.loc[has_ev, 'EV/EBITDA'].std()
    )
    df.loc[has_ev,  '종합점수'] = df.loc[has_ev,  ['z_per','z_pbr','z_roe','z_ev']].sum(axis=1)
    df.loc[~has_ev, '종합점수'] = df.loc[~has_ev, ['z_per','z_pbr','z_roe']].sum(axis=1)
    return df.sort_values('종합점수', ascending=False).reset_index(drop=True)

r_final = calc_final_score(result_final)
print(f'최종 종목 수: {len(r_final)}개')
r_final[['종목명','시장','PER','PBR','ROE','EV/EBITDA','시가총액_억','종합점수']].head(10)

## 7. 시각화

In [ ]:
# 시각화 1: PER·ROE 분포
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('KOSPI / KOSDAQ 밸류에이션 분포', fontsize=14,
             fontweight='bold', fontproperties=font_prop)
COLORS = {'KOSPI': '#2563EB', 'KOSDAQ': '#F97316'}
for mkt, color in COLORS.items():
    sub = df[df['시장'] == mkt]
    axes[0].hist(sub['PER'].clip(0,80), bins=60, alpha=0.5, color=color,
                 label=f'{mkt} (중앙값 {sub["PER"].median():.1f})')
    axes[1].hist(sub['ROE'].clip(0,40), bins=60, alpha=0.5, color=color,
                 label=f'{mkt} (중앙값 {sub["ROE"].median():.1f})')
for ax, title in zip(axes, ['PER 분포', 'ROE 분포 (%)']):
    ax.set_title(title, fontproperties=font_prop, fontweight='bold')
    ax.legend(prop=font_prop)
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('01_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 시각화 2: 스크리닝 결과 상위 20개
top20 = r_final.head(20)
bar_colors = ['#2563EB' if m=='KOSPI' else '#F97316' for m in top20['시장']]
fig, ax = plt.subplots(figsize=(12, 9))
ax.barh(top20['종목명'][::-1], top20['종합점수'][::-1],
        color=bar_colors[::-1], alpha=0.85)
for i, (_, row) in enumerate(top20[::-1].iterrows()):
    ev_str = f" | EV/EBITDA {row['EV/EBITDA']:.1f}" if pd.notna(row['EV/EBITDA']) else ''
    ax.text(row['종합점수']+0.03, i,
            f"PER {row['PER']:.1f} | PBR {row['PBR']:.2f} | ROE {row['ROE']:.1f}%{ev_str}",
            va='center', fontsize=7.5, fontproperties=font_prop)
ax.set_xlabel('종합점수 (Z-score 합산)', fontproperties=font_prop)
ax.set_title('최종 스크리닝 결과 상위 20개\n(PER<=15, ROE>=10%, 시가총액>=500억 | PBR·EV/EBITDA 반영)',
             fontproperties=font_prop, fontsize=12)
for label in ax.get_yticklabels():
    label.set_fontproperties(font_prop)
legend = [mpatches.Patch(color='#2563EB', label='KOSPI'),
          mpatches.Patch(color='#F97316', label='KOSDAQ')]
ax.legend(handles=legend, prop=font_prop)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('02_screened_top20.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 시각화 3: PER-ROE 인터랙티브 산점도
fig3 = px.scatter(
    r_final,
    x='PER', y='ROE',
    color='시장',
    size='시가총액_억',
    hover_name='종목명',
    hover_data={'PER':':.1f','PBR':':.2f','ROE':':.1f','EV/EBITDA':':.1f','시가총액_억':':,.0f'},
    title='스크리닝 통과 종목: PER vs ROE (버블: 시가총액)',
    color_discrete_map={'KOSPI':'#2563EB','KOSDAQ':'#F97316'},
    size_max=40, opacity=0.7,
)
fig3.update_layout(height=550)
fig3.show()

## 8. 결과 저장

In [ ]:
output_cols = ['종목명','시장','PER','PBR','ROE','EV/EBITDA','시가총액_억','종합점수']
r_final[output_cols].to_csv('screened_final.csv', index=False, encoding='utf-8-sig')

print('=' * 50)
print('  밸류에이션 스크리너 결과 요약')
print('=' * 50)
print(f'  분석 대상: KOSPI + KOSDAQ {len(df):,}개 종목')
print(f'  스크리닝 통과: {len(r_final)}개 종목')
print(f'  EV/EBITDA 산출: {r_final["EV/EBITDA"].notna().sum()}개 종목')
print()
print('[상위 10개 종목]')
print(r_final[output_cols].head(10).to_string(index=False))
print()
print('저장 완료 → screened_final.csv')